# Initialize MODIS Snow Phenology

This notebook does two things in order:

**Phase 1 — Tile Selection**: Downloads the MODIS 36×18 sinusoidal tile grid, intersects with a land mask, and writes `processing/tile_data/tile_processing_status.geojson` — the tile registry used by all downstream processing.

**Phase 2 — Icechunk Store Creation**: Creates an empty Zarr v3 store (with ShardingCodec) on Azure Blob Storage via Icechunk, ready to receive tile writes from GitHub Actions.

---
Run Phase 1 first. Edit the GeoJSON to manually mark any tiles you want to skip before running Phase 2 or any batch jobs.

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import geodatasets
import xarray as xr
import dask
import zarr
import icechunk
import rioxarray
import matplotlib.pyplot as plt
from pathlib import Path

import sys
sys.path.insert(0, str(Path('..')))
from modis_snow_phenology import masking, Config

config = Config()
REPO_ROOT = Path('..')
TILE_STATUS_PATH = REPO_ROOT / 'processing' / 'tile_data' / 'tile_processing_status.geojson'

## Phase 1: Tile Selection

Build the `tile_processing_status.geojson` GeoDataFrame with one row per MODIS tile.

In [ ]:
# Download MODIS sinusoidal tile grid (648 tiles: 36h × 18v)
modis_grid = gpd.read_file(
    'zip+http://book.ecosens.org/wp-content/uploads/2016/06/modis_grid.zip'
    '!modis_sinusoidal_grid_world.shp'
)
print(f'MODIS grid: {len(modis_grid)} tiles, CRS: {modis_grid.crs}')
modis_grid.head()

In [ ]:
# Load Natural Earth land mask and reproject to MODIS sinusoidal CRS
land = gpd.read_file(geodatasets.get_url('naturalearth land'))
land_modis_crs = gpd.GeoSeries(land.union_all(), crs='EPSG:4326').to_crs(modis_grid.crs)

# Determine which tiles intersect land
land_mask = modis_grid.intersects(land_modis_crs.union_all())

# Tile index 600 (h24v16) is a mostly-ocean tile near Antarctica — skip it
# Add any other manual exclusions here
MANUAL_SKIP_INDICES = {600}
for idx in MANUAL_SKIP_INDICES:
    land_mask.iloc[idx] = False

print(f'Land tiles: {land_mask.sum()}, Ocean/skip tiles: {(~land_mask).sum()}')

In [ ]:
# Build GeoDataFrame: reproject to WGS84 for GeoJSON compliance
gdf = modis_grid[['h', 'v', 'geometry']].copy()
gdf = gdf.to_crs('EPSG:4326')

# Add tile ID string (e.g., "h10v04")
gdf['tile'] = gdf.apply(lambda row: f"h{int(row['h']):02d}v{int(row['v']):02d}", axis=1)
gdf['h'] = gdf['h'].astype(int)
gdf['v'] = gdf['v'].astype(int)
gdf['land'] = land_mask.values

# Set initial processing status
gdf['processing_status'] = 'skip'
gdf.loc[gdf['land'], 'processing_status'] = 'unprocessed'

# Notes field: explain skipped tiles; user can add manual skip reasons here
gdf['notes'] = ''
gdf.loc[~gdf['land'], 'notes'] = 'Ocean tile'
for idx in MANUAL_SKIP_INDICES:
    tile = f"h{int(modis_grid.iloc[idx]['h']):02d}v{int(modis_grid.iloc[idx]['v']):02d}"
    gdf.loc[gdf['tile'] == tile, 'notes'] = 'Manually excluded: mostly ocean'

# Reorder columns
gdf = gdf[['tile', 'h', 'v', 'land', 'processing_status', 'notes', 'geometry']]

print(gdf['processing_status'].value_counts())
gdf.head(10)

In [ ]:
# Plot tile map
fig, ax = plt.subplots(figsize=(16, 8))
color_map = {'unprocessed': 'steelblue', 'skip': 'lightgray'}
for status, color in color_map.items():
    gdf[gdf['processing_status'] == status].plot(ax=ax, color=color, edgecolor='white', linewidth=0.3, label=status)
ax.set_title('MODIS Tile Processing Status\n(blue = land/unprocessed, gray = skip)', fontsize=14)
ax.legend()
ax.set_axis_off()
plt.tight_layout()

In [ ]:
# Save GeoJSON tile registry
TILE_STATUS_PATH.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(TILE_STATUS_PATH, driver='GeoJSON')
print(f'Saved {len(gdf)} tile records to {TILE_STATUS_PATH}')

---
## Phase 2: Icechunk Store Creation

Creates an empty Zarr v3 store on Azure Blob Storage using Icechunk.
The store is metadata-only at this point — tile data is written by GitHub Actions.

Requires: `AZURE_STORAGE_ACCOUNT` and `AZURE_STORAGE_SAS_TOKEN` environment variables.

In [ ]:
# Get MODIS sinusoidal grid x/y coordinates from STAC
# This fetches one MODIS scene to extract the full coordinate arrays
print('Fetching MODIS grid coordinates from Planetary Computer STAC...')
modis_full_grid = masking.get_modis_MOD10A2_full_grid()
y_coords = modis_full_grid.y.values
x_coords = modis_full_grid.x.values
print(f'Grid shape: y={len(y_coords)}, x={len(x_coords)}')
print(f'y range: [{y_coords.min():.1f}, {y_coords.max():.1f}]')
print(f'x range: [{x_coords.min():.1f}, {x_coords.max():.1f}]')

In [ ]:
# Build empty xarray template dataset
water_years = np.arange(config.wy_start, config.wy_end + 1)
fill_value = np.iinfo(np.int16).min  # -32768

shape = (len(water_years), len(y_coords), len(x_coords))
print(f'Store shape: {shape} ({np.prod(shape) * 2 / 1e9:.1f} GB uncompressed per variable)')

ds_template = xr.Dataset(
    {
        'SAD_DOWY': xr.DataArray(
            dask.array.full(shape, fill_value=fill_value, chunks=(1, 2400, 2400), dtype=np.int16),
            dims=('water_year', 'y', 'x'),
        ),
        'SDD_DOWY': xr.DataArray(
            dask.array.full(shape, fill_value=fill_value, chunks=(1, 2400, 2400), dtype=np.int16),
            dims=('water_year', 'y', 'x'),
        ),
        'max_consec_snow_days': xr.DataArray(
            dask.array.full(shape, fill_value=fill_value, chunks=(1, 2400, 2400), dtype=np.int16),
            dims=('water_year', 'y', 'x'),
        ),
    },
    coords={
        'water_year': water_years,
        'y': y_coords,
        'x': x_coords,
    },
)

# Add CRS (GeoZarr convention: spatial_ref variable + grid_mapping attribute)
modis_crs = modis_full_grid.rio.crs
ds_template = ds_template.rio.write_crs(modis_crs)
for var in ds_template.data_vars:
    ds_template[var].attrs['grid_mapping'] = 'spatial_ref'

# Add metadata
ds_template.water_year.attrs['description'] = (
    'Water year. Northern hemisphere: Oct 1 – Sep 30 (e.g. WY2015 = 2014-10-01 to 2015-09-30). '
    'Southern hemisphere: Apr 1 – Mar 31 (e.g. WY2015 = 2015-04-01 to 2016-03-31).'
)
ds_template.attrs['title'] = 'Global MODIS Snow Phenology'
ds_template.attrs['description'] = (
    'Snow appearance date (SAD), snow disappearance date (SDD), and maximum consecutive snow days '
    'derived from MODIS MOD10A2 8-day maximum snow extent product. '
    'Cloud filling follows Wrzesien et al. 2019.'
)
ds_template.attrs['source'] = 'MODIS MOD10A2.061 via Microsoft Planetary Computer'
ds_template.attrs['Conventions'] = 'CF-1.8'

for var in ['SAD_DOWY', 'SDD_DOWY']:
    ds_template[var].attrs['units'] = 'day of water year'
    ds_template[var].attrs['_FillValue'] = fill_value
ds_template['max_consec_snow_days'].attrs['units'] = 'days'
ds_template['max_consec_snow_days'].attrs['_FillValue'] = fill_value

ds_template

In [ ]:
# Create Icechunk repository on Azure Blob Storage
storage = config.get_storage()

# Uncomment when ready to create (this is irreversible — creates the store)
# repo = config.create_repo()

# To open an existing repo instead:
# repo = config.open_repo()

print('Storage config ready. Uncomment repo = config.create_repo() to create the store.')

In [ ]:
# Write empty template to the Icechunk store
# Shard: (1, 2400, 2400) = one storage object per water year per MODIS tile
# Inner chunk: (1, 600, 600) = 16 inner chunks per shard (2400/600=4, clean divisibility)

# Uncomment after creating/opening the repo above

# session = repo.writable_session('main')
# store = session.store()
#
# shard_codec = zarr.codecs.ShardingCodec(
#     chunk_shape=(1, 600, 600),
#     codecs=[
#         zarr.codecs.BytesCodec(),
#         zarr.codecs.BloscCodec(cname='zstd', clevel=5),
#     ],
# )
#
# data_vars = ['SAD_DOWY', 'SDD_DOWY', 'max_consec_snow_days']
# encoding = {
#     var: {
#         'chunks': (1, 2400, 2400),   # shard shape (outer)
#         'codecs': [shard_codec],
#         'fill_value': fill_value,
#         'dtype': 'int16',
#     }
#     for var in data_vars
# }
#
# ds_template.to_zarr(
#     store,
#     mode='w',
#     zarr_format=3,
#     encoding=encoding,
#     compute=False,
#     write_empty_chunks=False,
# )
#
# snapshot_id = session.commit('Initialize store: empty template, WY2015-2024')
# print(f'Store initialized. Snapshot ID: {snapshot_id}')

In [ ]:
# Verify store (uncomment after creating)

# session_ro = repo.readonly_session('main')
# ds_verify = xr.open_zarr(session_ro.store(), zarr_format=3, consolidated=False)
# print(ds_verify)
# print('CRS:', ds_verify.rio.crs)